In [0]:
from pyspark.sql import functions as F
dbutils.widgets.text("catalog", "uc_merchandise_lakehouse")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("storage_account", "stsarikacloud")
dbutils.widgets.text("raw_container", "external-tables")
dbutils.widgets.text("checkpoint_container", "checkpoints")
dbutils.widgets.text("source_folder", "e_commerce_sales")
 


In [0]:
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
storage_account = dbutils.widgets.get("storage_account")
raw_container = dbutils.widgets.get("raw_container")
checkpoint_container = dbutils.widgets.get("checkpoint_container")
source_folder = dbutils.widgets.get("source_folder")

In [0]:
source_path = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/{source_folder}/"
checkpoint_path = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/bronze/{source_folder}/_checkpoint/"
schema_location = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/bronze/{source_folder}/_schema/"
bad_records_path = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/bronze/{source_folder}/_bad_records/"

In [0]:
target_table = f"{catalog}.{bronze_schema}.online_sales_rawdata"
 
print(f"Source path      : {source_path}")
print(f"Checkpoint path  : {checkpoint_path}")
print(f"Schema location  : {schema_location}")
print(f"Target table     : {target_table}")

In [0]:
from pyspark.sql import functions as F

raw_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("badRecordsPath", bad_records_path)
    .option("multiLine", "true")
    .load(source_path)
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
    .withColumn("_source_system", F.lit("online_ecommerce"))
)

In [0]:
(raw_stream_df.writeStream
   .format("delta")
   .option("checkpointLocation", checkpoint_path)
   .option("mergeSchema", "true")
   .trigger(availableNow=True)   # process what's currently available, then stop
   .toTable(target_table)
)

In [0]:

%sql
select * from uc_merchandise_lakehouse.bronze.online_sales_rawdata